# EfficientNet Baseline — Fraud Detection

EfficientNet-B0 (default) fine-tuned for binary document/image fraud classification.

**Why EfficientNet vs ResNet-18?**  
EfficientNet uses compound scaling — width, depth, and resolution are scaled together
using a fixed ratio, making it more parameter-efficient than simply stacking more layers.
B0 achieves higher ImageNet accuracy than ResNet-18 with fewer parameters (~5.3M vs ~11M).

**Key differences from the ResNet-18 pipeline:**
- `model.classifier` instead of `model.fc`
- BatchNorm layers kept in **eval mode during Stage 2** (`freeze_bn=True`) to preserve ImageNet statistics
- `model.features[-1]` as GradCAM target layer
- `variant` parameter lets you swap B0 → B3 if B0 plateaus

**Pipeline:** Load splits → class balance → dataloaders → Stage 1 → Stage 2 → training curves → threshold tuning → test evaluation → score distribution → multi-seed → subgroup analysis → GradCAM → comparison with ResNet-18

## 0 · Imports & reproducibility

In [1]:
import sys, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc as sk_auc

sys.path.insert(0, r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\src")

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

from efficientnet_utils import (
    EFFICIENTNET_VARIANTS,
    load_image_splits, print_split_summary, build_dataloaders,
    build_val_transform, build_model, unfreeze_backbone, make_loss_fn,
    fit_model, run_test_evaluation, predict_probs,
    compute_metrics, tune_threshold, evaluate,
    subgroup_metrics, GradCAM,
)

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False

SEED   = 42
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}  |  PyTorch {torch.__version__}")

Device : cpu  |  PyTorch 2.11.0+cpu


## 1 · Paths & config

In [ ]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR     = PROJECT_ROOT / "data" / "processed"

# ── Change VARIANT here to try B1, B2, B3 ─────────────────────────────────
VARIANT = "b0"   # "b0" | "b1" | "b2" | "b3" | "b4"
# ──────────────────────────────────────────────────────────────────────────

_, _, IMAGE_SIZE = EFFICIENTNET_VARIANTS[VARIANT]
RESULTS_DIR = PROJECT_ROOT / "notebook" / "results" / f"efficientnet_{VARIANT}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATA_DIR / "image_train_group_test.csv"
VAL_CSV   = DATA_DIR / "image_val_group_test.csv"
TEST_CSV  = DATA_DIR / "image_test_group_test.csv"

BATCH_SIZE  = 32
NUM_WORKERS = 0

# Training hyper-parameters
STAGE1_EPOCHS   = 5
STAGE1_LR       = 1e-3
STAGE1_PATIENCE = 5

STAGE2_EPOCHS   = 20
BACKBONE_LR     = 1e-5
HEAD_LR         = 1e-4
STAGE2_PATIENCE = 6

print(f"Model    : EfficientNet-{VARIANT.upper()}")
print(f"Img size : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Results  : {RESULTS_DIR}")

## 2 · Load & validate splits

In [ ]:
splits = load_image_splits(train_csv=TRAIN_CSV, val_csv=VAL_CSV, test_csv=TEST_CSV)
print_split_summary(splits.train_df, splits.val_df, splits.test_df)

## 3 · Class balance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colours   = ["#4C72B0", "#DD8452"]

for ax, (name, df) in zip(axes, [("Train", splits.train_df),
                                   ("Val",   splits.val_df),
                                   ("Test",  splits.test_df)]):
    counts = df["label_int"].value_counts().sort_index()
    bars   = ax.bar(["Genuine","Fraud"], counts.values,
                    color=colours, edgecolor="white", width=0.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
                str(val), ha="center", va="bottom", fontsize=10)
    ax.set_title(f"{name}  (n={len(df):,})")
    ax.set_ylabel("Count"); ax.spines[["top","right"]].set_visible(False)

fig.suptitle("Class distribution across splits", fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "class_balance.png", dpi=150, bbox_inches="tight")
plt.show()

## 4 · Build dataloaders
The image size is set automatically to match the variant (224 for B0, 300 for B3). Using the wrong size hurts accuracy because EfficientNet compound scaling is calibrated to specific resolutions.

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(
    splits=splits, variant=VARIANT,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
)
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

## 5 · Stage 1 — frozen backbone (head warmup)

Same rationale as ResNet: train only the head first to avoid large random gradients
corrupting pretrained features. `freeze_bn=False` here because the backbone is frozen —
BatchNorm is already not updating its running stats.

In [ ]:
model   = build_model(variant=VARIANT, pretrained=True,
                      freeze_backbone=True, dropout=0.4).to(device)
loss_fn = make_loss_fn(splits.train_df, device=device)

optimizer_s1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=STAGE1_LR, weight_decay=1e-4,
)

model, history_s1 = fit_model(
    model=model, train_loader=train_loader, val_loader=val_loader,
    loss_fn=loss_fn, optimizer=optimizer_s1, device=device,
    epochs=STAGE1_EPOCHS, threshold=0.5,
    early_stopping_patience=STAGE1_PATIENCE,
    monitor_metric="val_roc_auc",
    freeze_bn=False,
    model_save_path=RESULTS_DIR / "stage1_best.pt",
)

print("\nStage 1 history:")
display(history_s1.round(4))

## 6 · Stage 2 — full fine-tuning (backbone unfrozen)

**EfficientNet-specific:** `freeze_bn=True` is passed to `fit_model`.
This keeps all BatchNorm layers in eval mode during Stage 2 training, which
preserves the well-calibrated running mean/variance statistics from ImageNet
pretraining. Without this, BN stats would be corrupted by your small dataset,
causing training instability and worse generalisation.

Differential LR still applies: backbone `1e-5`, head `1e-4`.

In [ ]:
unfreeze_backbone(model)

backbone_params = [p for n, p in model.named_parameters() if "classifier" not in n]
head_params     = list(model.classifier.parameters())

optimizer_s2 = torch.optim.Adam(
    [{"params": backbone_params, "lr": BACKBONE_LR},
     {"params": head_params,     "lr": HEAD_LR}],
    weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_s2, T_max=STAGE2_EPOCHS, eta_min=1e-7,
)

model, history_s2 = fit_model(
    model=model, train_loader=train_loader, val_loader=val_loader,
    loss_fn=loss_fn, optimizer=optimizer_s2, device=device,
    epochs=STAGE2_EPOCHS, threshold=0.5,
    early_stopping_patience=STAGE2_PATIENCE,
    monitor_metric="val_roc_auc",
    scheduler=scheduler,
    freeze_bn=True,
    model_save_path=RESULTS_DIR / "stage2_best.pt",
)

print("\nStage 2 history:")
display(history_s2.round(4))

## 7 · Training curves

In [ ]:
history = pd.concat([history_s1, history_s2], ignore_index=True)
history["epoch_global"] = range(1, len(history) + 1)
history.to_csv(RESULTS_DIR / "training_history.csv", index=False)

n_s1 = len(history_s1)
ep   = history["epoch_global"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(ep, history["train_loss"], label="train", color="#4C72B0")
axes[0].plot(ep, history["val_loss"],   label="val",   color="#DD8452")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(ep, history["val_f1"],      label="F1",      color="#55A868")
axes[1].plot(ep, history["val_roc_auc"], label="ROC-AUC", color="#C44E52")
axes[1].set_ylim(0.75, 1.01)
axes[1].set_title("Validation metrics"); axes[1].set_xlabel("Epoch"); axes[1].legend()

axes[2].plot(ep, history["lr"], color="#8172B2")
axes[2].set_title("Learning rate"); axes[2].set_xlabel("Epoch")
axes[2].set_yscale("log")

for ax in axes:
    ax.axvline(x=n_s1 + 0.5, color="gray", linestyle="--", alpha=0.7)
    ax.spines[["top","right"]].set_visible(False)
axes[0].text(n_s1/2, axes[0].get_ylim()[1]*0.99,
             "Stage 1", ha="center", va="top", fontsize=9, color="gray")
axes[0].text(n_s1+(len(ep)-n_s1)/2, axes[0].get_ylim()[1]*0.99,
             "Stage 2", ha="center", va="top", fontsize=9, color="gray")

fig.suptitle(f"EfficientNet-{VARIANT.upper()} training overview", fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 8 · Threshold tuning on the validation set

In [ ]:
val_y_true, val_y_prob = predict_probs(model, val_loader, device)
best_threshold, sweep_df = tune_threshold(val_y_true, val_y_prob, metric="f1")
sweep_df.to_csv(RESULTS_DIR / "threshold_sweep.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(sweep_df["threshold"], sweep_df["precision"], label="Precision", color="#4C72B0")
axes[0].plot(sweep_df["threshold"], sweep_df["recall"],    label="Recall",    color="#DD8452")
axes[0].plot(sweep_df["threshold"], sweep_df["f1"],        label="F1",        color="#55A868", lw=2)
axes[0].axvline(x=best_threshold, color="red", linestyle="--",
                label=f"Best t={best_threshold:.2f}")
axes[0].set_xlabel("Threshold"); axes[0].set_title("Metrics vs threshold")
axes[0].legend(); axes[0].spines[["top","right"]].set_visible(False)

fpr, tpr, _ = roc_curve(val_y_true, val_y_prob)
axes[1].plot(fpr, tpr, color="#C44E52", lw=2,
             label=f"Val ROC-AUC = {sk_auc(fpr,tpr):.4f}")
axes[1].plot([0,1],[0,1],"k--",lw=1)
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
axes[1].set_title("ROC Curve (validation)")
axes[1].legend(); axes[1].spines[["top","right"]].set_visible(False)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "threshold_and_roc_val.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Selected threshold: {best_threshold:.2f}")

## 9 · Test evaluation

In [ ]:
test_metrics = run_test_evaluation(
    model=model, test_loader=test_loader,
    loss_fn=loss_fn, device=device, threshold=best_threshold,
)

y_true_test, y_prob_test = predict_probs(model, test_loader, device)
test_pred_df = splits.test_df.reset_index(drop=True).copy()
test_pred_df["y_true"] = y_true_test
test_pred_df["y_prob"] = y_prob_test
test_pred_df["y_pred"] = (test_pred_df["y_prob"] >= best_threshold).astype(int)
test_pred_df.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metric_keys = ["accuracy","precision","recall","f1","roc_auc"]
metric_vals = [test_metrics[k] for k in metric_keys]
bar_colours = ["#4C72B0","#55A868","#DD8452","#C44E52","#8172B2"]
bars = axes[0].bar(metric_keys, metric_vals, color=bar_colours, edgecolor="white")
for bar, val in zip(bars, metric_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=9)
axes[0].set_ylim(0,1.12)
axes[0].set_title(f"EfficientNet-{VARIANT.upper()} test metrics")
axes[0].spines[["top","right"]].set_visible(False)

cm = np.array(test_metrics["confusion_matrix"])
axes[1].imshow(cm, cmap="Blues")
axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
axes[1].set_xticklabels(["Pred Genuine","Pred Fraud"])
axes[1].set_yticklabels(["True Genuine","True Fraud"])
axes[1].set_title(f"Confusion matrix  (t={best_threshold:.2f})")
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(cm[i,j]), ha="center", va="center",
                     fontsize=14,
                     color="white" if cm[i,j]>cm.max()/2 else "black")

fpr_t, tpr_t, _ = roc_curve(y_true_test, y_prob_test)
axes[2].plot(fpr_t, tpr_t, color="#C44E52", lw=2,
             label=f"Test ROC-AUC = {sk_auc(fpr_t,tpr_t):.4f}")
axes[2].plot([0,1],[0,1],"k--",lw=1)
axes[2].set_xlabel("FPR"); axes[2].set_ylabel("TPR")
axes[2].set_title("ROC Curve (test)"); axes[2].legend()
axes[2].spines[["top","right"]].set_visible(False)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "test_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

### Score distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(y_prob_test[y_true_test==0], bins=40, alpha=0.65,
        color="#4C72B0", label="Genuine", density=True)
ax.hist(y_prob_test[y_true_test==1], bins=40, alpha=0.65,
        color="#DD8452", label="Fraud",   density=True)
ax.axvline(x=best_threshold, color="red", linestyle="--",
           label=f"Threshold = {best_threshold:.2f}")
ax.set_xlabel("Predicted fraud probability"); ax.set_ylabel("Density")
ax.set_title(f"Score distribution — EfficientNet-{VARIANT.upper()} test set")
ax.legend(); ax.spines[["top","right"]].set_visible(False)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 10 · Multi-seed robustness

In [ ]:
SEED_LIST    = [42, 52, 62]
seed_results = []

for seed in SEED_LIST:
    print("\n" + "="*55 + f"  SEED {seed}  " + "="*55)
    set_seed(seed)
    s_tl, s_vl, s_tel = build_dataloaders(splits=splits, variant=VARIANT,
                                           batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    # Stage 1
    s_m  = build_model(variant=VARIANT, pretrained=True,
                        freeze_backbone=True, dropout=0.4).to(device)
    s_l  = make_loss_fn(splits.train_df, device=device)
    s_o1 = torch.optim.Adam(filter(lambda p: p.requires_grad, s_m.parameters()),
                              lr=1e-3, weight_decay=1e-4)
    s_m, _ = fit_model(model=s_m, train_loader=s_tl, val_loader=s_vl,
                        loss_fn=s_l, optimizer=s_o1, device=device,
                        epochs=STAGE1_EPOCHS, early_stopping_patience=STAGE1_PATIENCE,
                        monitor_metric="val_roc_auc", freeze_bn=False)
    # Stage 2
    unfreeze_backbone(s_m)
    s_bp = [p for n, p in s_m.named_parameters() if "classifier" not in n]
    s_hp = list(s_m.classifier.parameters())
    s_o2 = torch.optim.Adam([{"params":s_bp,"lr":BACKBONE_LR},
                               {"params":s_hp,"lr":HEAD_LR}], weight_decay=1e-4)
    s_sc = torch.optim.lr_scheduler.CosineAnnealingLR(s_o2, T_max=STAGE2_EPOCHS, eta_min=1e-7)
    s_m, s_hist = fit_model(model=s_m, train_loader=s_tl, val_loader=s_vl,
                              loss_fn=s_l, optimizer=s_o2, device=device,
                              epochs=STAGE2_EPOCHS, early_stopping_patience=STAGE2_PATIENCE,
                              monitor_metric="val_roc_auc", scheduler=s_sc,
                              freeze_bn=True,
                              model_save_path=RESULTS_DIR / f"best_seed_{seed}.pt")
    sv_y, sv_p = predict_probs(s_m, s_vl, device)
    s_t, _     = tune_threshold(sv_y, sv_p, metric="f1")
    s_test     = run_test_evaluation(model=s_m, test_loader=s_tel,
                                      loss_fn=s_l, device=device, threshold=s_t)
    best_row   = s_hist.loc[s_hist["val_roc_auc"].idxmax()]
    seed_results.append({
        "seed": seed, "best_epoch": int(best_row["epoch"]),
        "tuned_threshold": round(s_t, 2),
        "test_accuracy":   round(float(s_test["accuracy"]),  4),
        "test_precision":  round(float(s_test["precision"]), 4),
        "test_recall":     round(float(s_test["recall"]),    4),
        "test_f1":         round(float(s_test["f1"]),        4),
        "test_roc_auc":    round(float(s_test["roc_auc"]),   4),
    })

In [ ]:
seed_df = pd.DataFrame(seed_results)
display(seed_df)

summary = seed_df[["test_accuracy","test_precision","test_recall",
                    "test_f1","test_roc_auc"]].agg(["mean","std"]).round(4)
print("\nMean ± std across seeds:")
display(summary)
seed_df.to_csv(RESULTS_DIR / "seed_results.csv", index=False)

metrics_to_plot = ["test_accuracy","test_precision","test_recall","test_f1","test_roc_auc"]
x     = np.arange(len(metrics_to_plot))
width = 0.22
clrs  = ["#4C72B0","#DD8452","#55A868"]

fig, ax = plt.subplots(figsize=(9,4))
for i, (_, row) in enumerate(seed_df.iterrows()):
    vals = [row[m] for m in metrics_to_plot]
    ax.bar(x+i*width, vals, width,
           label=f"Seed {int(row['seed'])}", color=clrs[i], alpha=0.85)
ax.set_xticks(x+width); ax.set_xticklabels(["Acc","Prec","Recall","F1","ROC-AUC"])
ax.set_ylim(0.75,1.05)
ax.set_title(f"EfficientNet-{VARIANT.upper()} — test metrics across seeds")
ax.set_ylabel("Score"); ax.legend()
ax.spines[["top","right"]].set_visible(False)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "seed_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 11 · Subgroup analysis

In [ ]:
def plot_subgroup(sg_df, group_col, ax_f1, ax_rc):
    df = sg_df.sort_values("f1", ascending=True)
    get_c = lambda v: "#55A868" if v>=0.85 else "#DD8452" if v>=0.70 else "#C44E52"
    ax_f1.barh(df[group_col].astype(str), df["f1"],
               color=[get_c(v) for v in df["f1"]], edgecolor="white")
    ax_f1.axvline(x=0.85, color="gray", linestyle="--", alpha=0.5)
    ax_f1.set_xlabel("F1"); ax_f1.set_xlim(0,1.05)
    ax_f1.spines[["top","right"]].set_visible(False)
    ax_rc.barh(df[group_col].astype(str), df["recall"],
               color=[get_c(v) for v in df["recall"]], edgecolor="white")
    ax_rc.axvline(x=0.85, color="gray", linestyle="--", alpha=0.5)
    ax_rc.set_xlabel("Recall"); ax_rc.set_xlim(0,1.05)
    ax_rc.spines[["top","right"]].set_visible(False)

for col in ["source_dataset","source_type","doc_type"]:
    if col not in test_pred_df.columns:
        print(f"Column '{col}' not found — skipping.")
        continue
    sg = subgroup_metrics(test_pred_df, col, threshold=best_threshold)
    sg.to_csv(RESULTS_DIR / f"test_metrics_by_{col}.csv", index=False)
    n = len(sg)
    fig, axes = plt.subplots(1, 2, figsize=(13, max(3, n*0.5+1)))
    plot_subgroup(sg, col, axes[0], axes[1])
    fig.suptitle(f"EfficientNet-{VARIANT.upper()} — subgroup metrics by {col}", fontsize=12)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"subgroup_{col}.png", dpi=150, bbox_inches="tight")
    plt.show()
    display(sg.round(3))

## 12 · GradCAM
**EfficientNet target layer:** `model.features[-1]` — the last MBConv block,
equivalent to `model.layer4[-1]` in ResNet.

In [ ]:
from PIL import Image as PILImage

_, _, img_sz  = EFFICIENTNET_VARIANTS[VARIANT]
cam           = GradCAM(model, target_layer=model.features[-1])
val_transform = build_val_transform(img_sz)

correct   = test_pred_df[test_pred_df["y_true"]==test_pred_df["y_pred"]].sample(
                n=min(4, (test_pred_df["y_true"]==test_pred_df["y_pred"]).sum()),
                random_state=SEED)
incorrect = test_pred_df[test_pred_df["y_true"]!=test_pred_df["y_pred"]].sample(
                n=min(4, (test_pred_df["y_true"]!=test_pred_df["y_pred"]).sum()),
                random_state=SEED)
sample_df = pd.concat([correct, incorrect]).reset_index(drop=True)

n    = len(sample_df)
fig, axes = plt.subplots(2, n, figsize=(n*2.8, 5.5))
if n == 1: axes = [[axes[0]], [axes[1]]]

for col, (_, row) in enumerate(sample_df.iterrows()):
    pil_img = PILImage.open(row["image_path"]).convert("RGB")
    tensor  = val_transform(pil_img).unsqueeze(0).to(device)
    heatmap = cam(tensor)
    overlay = GradCAM.overlay(pil_img, heatmap, alpha=0.45)

    true_lbl = "genuine" if int(row["y_true"])==0 else "fraud"
    pred_lbl = "genuine" if int(row["y_pred"])==0 else "fraud"
    ok       = true_lbl == pred_lbl

    axes[0][col].imshow(pil_img)
    axes[0][col].set_title(f"GT: {true_lbl}", fontsize=8)
    axes[0][col].axis("off")
    axes[1][col].imshow(overlay)
    axes[1][col].set_title(f"Pred: {pred_lbl}", fontsize=8,
                            color="#2ca02c" if ok else "#d62728")
    axes[1][col].axis("off")

fig.suptitle(f"EfficientNet-{VARIANT.upper()} GradCAM\n"
             "Original (top) | Heatmap (bottom)  ·  Green=correct  Red=wrong",
             fontsize=10)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "gradcam_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 13 · Comparison with ResNet-18 baseline

Load the ResNet-18 results CSV (produced by `08_image_baseline.ipynb`) and plot
both models side by side. Run this cell **after** you have run both notebooks.

In [ ]:
resnet_results_dir = PROJECT_ROOT / "notebook" / "results" / "image_baseline"
resnet_seed_csv    = resnet_results_dir / "seed_results.csv"
effnet_seed_csv    = RESULTS_DIR / "seed_results.csv"

if not resnet_seed_csv.exists():
    print("ResNet-18 seed_results.csv not found — run 08_image_baseline.ipynb first.")
else:
    rn_df  = pd.read_csv(resnet_seed_csv)
    eff_df = pd.read_csv(effnet_seed_csv)

    metrics_cmp = ["test_accuracy","test_precision","test_recall","test_f1","test_roc_auc"]
    rn_means    = rn_df[metrics_cmp].mean()
    eff_means   = eff_df[metrics_cmp].mean()
    rn_stds     = rn_df[metrics_cmp].std()
    eff_stds    = eff_df[metrics_cmp].std()

    x     = np.arange(len(metrics_cmp))
    width = 0.35

    fig, ax = plt.subplots(figsize=(11, 5))
    b1 = ax.bar(x - width/2, rn_means.values,  width, yerr=rn_stds.values,
                label="ResNet-18",                 color="#4C72B0", alpha=0.85,
                capsize=4, error_kw={"elinewidth":1.5})
    b2 = ax.bar(x + width/2, eff_means.values, width, yerr=eff_stds.values,
                label=f"EfficientNet-{VARIANT.upper()}", color="#DD8452", alpha=0.85,
                capsize=4, error_kw={"elinewidth":1.5})

    ax.set_xticks(x)
    ax.set_xticklabels(["Accuracy","Precision","Recall","F1","ROC-AUC"])
    ax.set_ylim(0.75, 1.08)
    ax.set_ylabel("Score (mean ± std, 3 seeds)")
    ax.set_title("ResNet-18 vs EfficientNet — Test metrics comparison")
    ax.legend(); ax.spines[["top","right"]].set_visible(False)

    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Summary table
    cmp_table = pd.DataFrame({
        "ResNet-18 mean":              rn_means.round(4).values,
        "ResNet-18 std":               rn_stds.round(4).values,
        f"EfficientNet-{VARIANT.upper()} mean": eff_means.round(4).values,
        f"EfficientNet-{VARIANT.upper()} std":  eff_stds.round(4).values,
    }, index=["Accuracy","Precision","Recall","F1","ROC-AUC"])
    display(cmp_table)
    cmp_table.to_csv(RESULTS_DIR / "model_comparison.csv")

## 14 · Final summary

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.axis("off")
rows_data = [[f"{test_metrics[k]:.4f}"]
             for k in ["accuracy","precision","recall","f1","roc_auc"]]
tbl = ax.table(
    cellText=rows_data,
    rowLabels=["Accuracy","Precision","Recall","F1","ROC-AUC"],
    colLabels=["Score"],
    cellLoc="center", rowLoc="center", loc="center",
)
tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1.2, 1.8)
ax.set_title(f"EfficientNet-{VARIANT.upper()} — Test Results\n"
             f"threshold={best_threshold:.2f}  |  seed={SEED}", fontsize=11, pad=18)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "final_summary_table.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nAll results saved to:", RESULTS_DIR)